Procesamiento de validaciones
En este notebook juntamos los archivos diarios de validaciones de TransMilenio en una sola tabla. Cada archivo tiene una fila por cada persona que pasó por un torniquete, así que lo resumimos para que quede una fila por estación, día y hora.
Lo hacemos por evento. Primero para Calle 53, que abrió el sábado 24 de octubre de 2026 y reemplazó a la estación temporal Marly. Para Calle 72, que abrió el 22 de agosto, solo hay que cambiar la configuración de abajo.

In [4]:
import pandas as pd

Configuración

Escribimos el nombre del evento, la fecha en que abrió la estación y la lista de días que vamos a usar. Usamos lunes a viernes de tres semanas antes y tres semanas después. Quitamos los festivos (12 de octubre y 2 de noviembre) y el 14 de octubre, porque ese día hubo marchas y la demanda fue mucho menor de lo normal.

In [5]:
evento = "calle53"
fecha_evento = pd.to_datetime("2026-10-24")
carpeta = "../data/raw/validaciones/calle53/"

fechas = [
    "20261005", "20261006", "20261007", "20261008", "20261009",
    "20261013", "20261015", "20261016",
    "20261019", "20261020", "20261021", "20261022", "20261023",
    "20261026", "20261027", "20261028", "20261029", "20261030",
    "20261103", "20261104", "20261105", "20261106",
    "20261109", "20261110", "20261111", "20261112", "20261113",
]

Leer y resumir cada día

Para cada fecha leemos el archivo con solo tres columnas: la hora de la validación, la estación y la troncal. Luego contamos cuántas validaciones hubo por estación y por hora.

Algunos archivos traen unas pocas validaciones con fecha de otro día, por eso nos quedamos solo con las del día del archivo. Esta celda puede tardar varios minutos.

In [6]:
resumenes = []

for fecha in fechas:
    df = pd.read_csv(carpeta + "validacionTroncal" + fecha + ".zip",
                     usecols=["Fecha_Transaccion", "Estacion_Parada", "Linea"])

    df["Fecha_Transaccion"] = pd.to_datetime(df["Fecha_Transaccion"])
    df["fecha"] = df["Fecha_Transaccion"].dt.normalize()
    df["hora"] = df["Fecha_Transaccion"].dt.hour
    df = df[df["fecha"] == pd.to_datetime(fecha)]

    df["codigo"] = df["Estacion_Parada"].str.strip().str[1:6]

    resumen = (df.groupby(["fecha", "codigo", "hora"])
                 .agg(validaciones=("codigo", "count"),
                      estacion=("Estacion_Parada", "first"),
                      linea=("Linea", "first"))
                 .reset_index())
    resumenes.append(resumen)

tabla = pd.concat(resumenes, ignore_index=True)
tabla.shape

(79452, 6)

Columnas nuevas

Una misma estación a veces aparece con dos nombres distintos, por eso usamos el código de la estación y le dejamos a cada código un solo nombre. También agregamos si el día es antes o después del evento y el día de la semana.

In [7]:
nombres = tabla.groupby("codigo")["estacion"].first()
lineas = tabla.groupby("codigo")["linea"].first()
tabla["estacion"] = tabla["codigo"].map(nombres).str.strip().str[8:]
tabla["linea"] = tabla["codigo"].map(lineas).str.strip()

tabla["evento"] = evento
tabla["periodo"] = "antes"
tabla.loc[tabla["fecha"] >= fecha_evento, "periodo"] = "después"

dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
tabla["dia_semana"] = tabla["fecha"].dt.dayofweek.map(lambda d: dias[d])

tabla.head()

,fecha,codigo,hora,validaciones,estacion,linea,evento,periodo,dia_semana
0,2026-10-05,02000,4,678,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle53,antes,lunes
1,2026-10-05,02000,5,4466,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle53,antes,lunes
2,2026-10-05,02000,6,8605,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle53,antes,lunes
3,2026-10-05,02000,7,8916,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle53,antes,lunes
4,2026-10-05,02000,8,5839,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle53,antes,lunes


Revisión rápida
Revisamos que estén todas las fechas y que no haya filas repetidas para la misma estación, día y hora.

In [8]:
tabla["fecha"].nunique(), tabla.duplicated(["fecha", "codigo", "hora"]).sum()

(27, np.int64(0))


Guardamos la tabla en la carpeta de datos procesados con el nombre del evento.

In [9]:
tabla.to_csv("../data/processed/validaciones_" + evento + ".csv", index=False)


En los datos, la nueva estación Calle 53 aparece con el nombre Marly - Calle 57 y el código 09131. La temporal Marly que cerró tiene el código 09126.
No usamos la columna Hora_Pico_SN porque dice Peak Time en todas las filas, incluso de madrugada.